# Recommender Baseline with Learned Embeddings (PyTorch)

This notebook implements an end-to-end **matrix-factorization-style** baseline for explicit ratings. It shows the pieces of a recommender:

- ID encoding → PyTorch datasets → mini-batching → embedding lookups → rating prediction → regression loss → ranking-style metrics.

---
Encoding IDs with `LabelEncoder`
- **What:** We map `userId` and `movieId` to contiguous integers starting at 0 using the scikit's `LabelEncoder`.
- **Why:** `nn.Embedding` expects indices in `[0, N-1]`. Real IDs are often large/sparse (e.g., 123456), which cannot be used as embedding indices directly.
- **Benefit:** Memory-efficient, fast lookups, and aligns IDs with embedding rows.

Note: if you ever see a “index out of range in Embedding” error, it means the IDs weren’t encoded or the cardinalities were miscounted.


In [11]:
# Imports
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from sklearn import model_selection, preprocessing
from sklearn.metrics import mean_squared_error


# Config
DATA_PATH = "ratings.csv"
BATCH_SIZE = 4
NUM_EPOCHS = 1
EMBEDDING_SIZE = 32
TEST_SIZE = 0.2
SEED = 42
TOP_K = 10
THRESHOLD = 3.5

In [12]:
# Data Loading
df = pd.read_csv(DATA_PATH)
print(f"Unique Users: {df.userId.nunique()}, Unique Movies: {df.movieId.nunique()}")

# Encode user and movie IDs starting from 0 - LabelEncoder converts categorical labels into a continuous range of integers starting from 0
lbl_user = preprocessing.LabelEncoder()
lbl_movie = preprocessing.LabelEncoder()
df["userId"] = lbl_user.fit_transform(df["userId"].values)
df["movieId"] = lbl_movie.fit_transform(df["movieId"].values)

# Train-test split
df_train, df_test = model_selection.train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, stratify=df["rating"].values
)

# Dataset Class
class MovieDataset(Dataset):
    def __init__(self, users, movies, ratings):
        self.users = users
        self.movies = movies
        self.ratings = ratings

    def __len__(self) -> int:
        return len(self.users)

    def __getitem__(self, idx: int):
        return (
            torch.tensor(self.users[idx], dtype=torch.long),
            torch.tensor(self.movies[idx], dtype=torch.long),
            torch.tensor(self.ratings[idx], dtype=torch.float32),
        )

train_dataset = MovieDataset(df_train.userId.values, df_train.movieId.values, df_train.rating.values)
valid_dataset = MovieDataset(df_test.userId.values, df_test.movieId.values, df_test.rating.values)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

Unique Users: 610, Unique Movies: 9724


In [ ]:
# Model Class
"""
Simple recommender model using embeddings for users and movies.

Architecture:
1. user_embed: Embedding layer mapping each user ID to a dense latent vector
2. movie_embed: Embedding layer mapping each movie ID to a dense latent vector
3. Linear layer (out): Takes the concatenated user+movie embeddings and outputs a single rating prediction.

Design notes:
- Embeddings capture latent characteristics of users and movies
- Concatenation allows the linear layer to learn how user and movie factors interact.
"""
class RecSysModel(nn.Module):
    def __init__(self, n_users: int, n_movies: int, n_embeddings: int = EMBEDDING_SIZE):
        super().__init__()
        self.user_embed = nn.Embedding(n_users, n_embeddings)
        self.movie_embed = nn.Embedding(n_movies, n_embeddings)
        # We take the n_emeddings and multiply by two because we have two embeddings (user and movie)
        self.out = nn.Linear(n_embeddings * 2, 1)

    """
    Forward pass through the model:
       - Looks up embeddings for users and movies
       - Concatenates embeddings
       - Passes through linear layer → predicts ratings
    """
    def forward(self, users, movies):
        user_embeds = self.user_embed(users)       # shape: (batch_size, embedding_dim)
        movie_embeds = self.movie_embed(movies)    # shape: (batch_size, embedding_dim)
        x = torch.cat([user_embeds, movie_embeds], dim=1)
        return self.out(x)

# Model Training
model = RecSysModel(len(lbl_user.classes_), len(lbl_movie.classes_))
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()

model.train()
for epoch in range(NUM_EPOCHS):
    for users, movies, ratings in train_loader:
        optimizer.zero_grad()
        y_pred = model(users, movies)
        loss = criterion(y_pred.squeeze(), ratings)
        loss.backward()
        optimizer.step()


In [17]:
# Model Evaluation
def evaluate(model, loader):
    model.eval() # tell the model we're in inference mode
    y_preds, y_trues = [], []
    with torch.no_grad():   # stop tracking gradients
        for users, movies, ratings in loader:
            preds = model(users, movies).squeeze()  # do a forward pass and then squeeze removes the extra dimension
            y_preds.extend(preds.tolist())          # extend just gets all of the pred list and puts it in y_pred
            y_trues.extend(ratings.tolist())
    return mean_squared_error(y_trues, y_preds)

mse = evaluate(model, test_loader)
print(f"Mean Squared Error: {mse:.4f} \n")

# Precision and Recall
"""
Intuition for these metrics:
 - Precision@K - Of the top K items I recommended, how many are actually good?
 - Recall@K - Of all the items a user likes, how many did I recommend in the top K?
"""
def precision_recall_at_k(model, loader, k=TOP_K, threshold=THRESHOLD):
    user_movie_test = defaultdict(list)

    with torch.no_grad():
        for users, movies, ratings in loader:
            preds = model(users, movies).squeeze()
            for u, m, y_pred, y_true in zip(users, movies, preds, ratings):
                user_movie_test[u.item()].append((y_pred.item(), y_true.item()))
            print(f"User ID: {u.item()}, Movie ID: {m.item()}, "
                f"Predicted rating: {y_pred.item():.2f}, True rating: {y_true.item():.2f}")

    precisions, recalls = {}, {}
    for uid, ratings in user_movie_test.items():
        ratings.sort(key=lambda x: x[0], reverse=True)

        n_rel = sum(r_true >= threshold for _, r_true in ratings)
        n_rec_k = sum(r_pred >= threshold for r_pred, _ in ratings[:k])
        n_rel_and_rec_k = sum(
            (r_true >= threshold and r_pred >= threshold)
            for r_pred, r_true in ratings[:k]
        )

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k > 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel > 0 else 0

    return (
        sum(precisions.values()) / len(precisions),
        sum(recalls.values()) / len(recalls),
    )

precision, recall = precision_recall_at_k(model, test_loader)
print(f"\nPrecision@{TOP_K}: {precision:.4f}")
print(f"Recall@{TOP_K}: {recall:.4f}")

Mean Squared Error: 0.8462 

User ID: 479, Movie ID: 260, Predicted rating: 3.07, True rating: 4.50
User ID: 482, Movie ID: 1070, Predicted rating: 3.17, True rating: 3.00
User ID: 604, Movie ID: 325, Predicted rating: 3.22, True rating: 4.00
User ID: 287, Movie ID: 1544, Predicted rating: 3.17, True rating: 3.00
User ID: 427, Movie ID: 4607, Predicted rating: 3.24, True rating: 4.50
User ID: 544, Movie ID: 1825, Predicted rating: 2.81, True rating: 4.00
User ID: 380, Movie ID: 507, Predicted rating: 4.07, True rating: 2.50
User ID: 524, Movie ID: 6659, Predicted rating: 3.65, True rating: 3.50
User ID: 429, Movie ID: 1796, Predicted rating: 3.82, True rating: 5.00
User ID: 437, Movie ID: 1170, Predicted rating: 3.22, True rating: 3.50
User ID: 114, Movie ID: 1070, Predicted rating: 3.60, True rating: 2.00
User ID: 90, Movie ID: 3958, Predicted rating: 3.22, True rating: 1.00
User ID: 519, Movie ID: 1489, Predicted rating: 3.66, True rating: 3.50
User ID: 17, Movie ID: 8045, Predicted 